<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semanas 02 a 05: Ejercicios de repaso 1</p>
  </div>
</div>

# Ejercicios de repaso 1 · Clases 2 a 5

Estos ejercicios repasan, con ejercicios prácticos, las clases **2** (conocer y preparar los datos), **3**
(regresión), **4** (clasificación) y **5** (no supervisado y segmentación). Son **actividades
formativas**: sirven para practicar y recibir *feedback*.

**Cómo trabajarlos:**

- Ejecuten primero las celdas de **preparación** y la celda de **datos** de cada bloque.
- En cada ejercicio hay una celda de código con **pistas** como comentarios: casi siempre basta con
  copiar la pista, quitarle el `#` y completarla. El código es corto a propósito (*low-code*).
- Después de cada ejercicio hay una celda **"Su lectura"**: lo central del módulo no es el código, sino
  **interpretar y decidir**. Respondan ahí con sus palabras.
- Cada bloque parte de cero (vuelve a cargar sus datos), así que pueden hacerlos en cualquier orden.
- Cuando terminen, comparen con el notebook de **solución comentada**: explica por qué se escribió cada
  parte del código y, en los ejercicios más complejos, entrega un *prompt* para pedirle a una IA que les
  ayude a corregir su propio código.

## Preparación del entorno

Instalamos las librerías (la salida queda oculta) y dejamos la configuración general.

In [ ]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets pyod "setuptools<81"

In [ ]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px

# en Colab los datos se leen del repo público; en local, desde ../datasets/
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

## Bloque A · Clase 2: conocer y preparar los datos

**Caso:** casas en venta de la Región Metropolitana (`precios_casas_rm.csv`). Cada fila es una casa;
`precio_uf` es su precio de venta. En este bloque **no entrenamos modelos**: conocemos la tabla, revisamos
su calidad y la dejamos lista.

In [ ]:
casas = pd.read_csv(BASE + 'precios_casas_rm.csv')
print('Filas y columnas:', casas.shape)
casas.head()

### Ejercicio A1 · Tipos de variables

Muestren cuántos **valores distintos** tiene cada columna. Con eso y con la vista de la tabla,
clasifiquen cada columna como **numérica** (discreta o continua) o **categórica** (nominal u ordinal), e
identifiquen el **target** y las **features**.

**Criterio de éxito:** las 15 columnas clasificadas, con una razón breve para cada caso dudoso.

In [ ]:
# pista: una sola línea. nunique() cuenta los valores distintos de cada columna.
# casas.nunique()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio A2 · Valores faltantes e imputación

Cuenten los **faltantes por columna**. Luego, **en una copia** de la tabla, rellenen los faltantes de
`area_construida` con la **mediana** y comprueben que esa columna quedó sin faltantes. ¿Con qué rellenarían
`estacionamiento`, que es Si/No?

**Criterio de éxito:** saber qué columnas tienen faltantes y cuántos, y explicar por qué cada columna se rellena distinto.

In [ ]:
# pista:
# casas.isna().sum()                         # conteo de faltantes por columna
# copia = casas.copy()
# copia['area_construida'] = copia['area_construida'].fillna(copia['area_construida'].median())
# copia['area_construida'].isna().sum()      # debería dar 0

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio A3 · Outliers con un boxplot

Dibujen un **boxplot** de `precio_uf` separado por `metro` (Si/No). Luego ordenen la tabla por precio
y muestren las **5 casas más caras**. Decidan: ¿son errores de datos o casos reales que el modelo debe
aprender?

**Criterio de éxito:** leer en el gráfico dónde están los outliers de cada grupo y argumentar una decisión con las 5 filas a la vista.

In [ ]:
# pista:
# px.box(casas, x='metro', y='precio_uf', points='outliers')
# casas.sort_values('precio_uf', ascending=False).head(5)

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio A4 · Dejar los datos listos con PyCaret

Creen un `RegressionExperiment` con target `precio_uf`, semilla `session_id=42` y un
**80% de los datos para entrenar** (`train_size=0.8`). Muestren cuántas filas quedaron en entrenamiento y
en prueba. Respondan: ¿qué hizo PyCaret automáticamente y qué decisiones siguen siendo de ustedes?
¿Podrían agregar una columna `precio_por_m2` como predictora? ¿Por qué?

**Criterio de éxito:** ver las dos particiones con sus tamaños y distinguir con claridad lo automático del criterio humano.

In [ ]:
# pista:
# from pycaret.tasks import RegressionExperiment
# exp = RegressionExperiment(target='precio_uf', session_id=42, train_size=0.8).fit(casas)
# exp.X_train.shape   y   exp.X_test.shape

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque B · Clase 3: regresión

**Caso:** el mismo de casas. Ahora sí entrenamos modelos para **predecir `precio_uf`** y aprendemos a leer
sus métricas en UF. Ejecuten la celda de partida: deja listo el experimento (70% entrenamiento,
validación cruzada de 5 pliegues).

In [ ]:
from pycaret.tasks import RegressionExperiment

casas = pd.read_csv(BASE + 'precios_casas_rm.csv')
exp_reg = RegressionExperiment(target='precio_uf', session_id=42, train_size=0.7, fold=5).fit(casas)
print('Entrenamiento:', exp_reg.X_train.shape, '| Prueba:', exp_reg.X_test.shape)

### Ejercicio B1 · Comparar modelos y leer el leaderboard

Comparen **regresión lineal** (`lr`), **árbol de decisión** (`dt`), ***random forest*** (`rf`) y
***gradient boosting*** (`gbr`), ordenando por **MAE**. Muestren la tabla y guarden el mejor modelo en
`mejor_reg`. Respondan: ¿qué modelo gana?, ¿cuánto se equivoca en promedio, en UF?, ¿cuánto peor es la
regresión lineal?

**Criterio de éxito:** identificar el ganador y traducir su MAE y su R² a una frase de negocio.

In [ ]:
# pista:
# res_reg = exp_reg.compare_models(include=['lr', 'dt', 'rf', 'gbr'], sort='MAE')
# res_reg.leaderboard     # tabla de líderes
# mejor_reg = res_reg.best

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio B2 · ¿Qué variables pesan?

Grafiquen la **importancia de variables** de `mejor_reg`. Respondan: ¿cuáles son las tres variables que
más pesan?, ¿les parece razonable desde el negocio inmobiliario?, ¿significa que esas variables
**causan** el precio?

**Criterio de éxito:** nombrar las variables principales y distinguir asociación de causalidad.

In [ ]:
# pista: exp_reg.plot_model(mejor_reg, plot='feature')

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio B3 · El examen final: el conjunto de prueba

Evalúen `mejor_reg` sobre el **conjunto de prueba** y comparen su MAE con el de la tabla de B1.
¿El modelo **generaliza** o hay señales de **sobreajuste**?

**Criterio de éxito:** poner lado a lado el MAE de la tabla de B1 y el de prueba, y concluir con esa comparación.

In [ ]:
# pista:
# pred_reg = exp_reg.predict_model(mejor_reg)
# pred_reg.metrics       # métricas sobre casas que el modelo nunca vio

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio B4 · ¿Dónde se equivoca más el modelo?

Con las predicciones de prueba, creen una columna con el **error absoluto** de cada casa y calculen el
**error promedio según `metro`** (Si/No), junto al precio promedio de cada grupo. ¿Dónde se equivoca más?
¿Qué le dirían a un usuario del modelo?

**Criterio de éxito:** una tabla con el error promedio por grupo y una advertencia concreta de uso del modelo.

In [ ]:
# pista:
# p = pred_reg.predictions                       # la predicción está en la columna 'prediction_label'
# p['error_abs'] = (p['precio_uf'] - p['prediction_label']).abs()
# p.groupby('metro')[['error_abs', 'precio_uf']].mean()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque C · Clase 4: clasificación

**Caso nuevo:** solicitudes de crédito de un banco (`credito_aleman.csv`, datos reales de UCI). Cada fila
es una persona que pidió un crédito; `incumplimiento` vale **1** si no lo pagó y **0** si lo pagó. Las
demás columnas ya vienen como números: las de montos y plazos son continuas, y `cuenta_corriente`,
`ahorros` e `historial_crediticio` son **códigos ordinales**.

**La decisión de negocio:** aprobar o rechazar el crédito.

In [ ]:
credito = pd.read_csv(BASE + 'credito_aleman.csv')
print('Filas y columnas:', credito.shape)
credito.head()

### Ejercicio C1 · El desbalance y la línea base ingenua

Calculen qué proporción de solicitudes terminó en **incumplimiento**. Si un modelo "ingenuo" aprobara
a **todos** (predijera siempre 0), ¿qué *accuracy* tendría? ¿Sería un buen modelo para el banco?

**Criterio de éxito:** la proporción de cada clase y el accuracy de la línea base, con una frase que explique por qué engaña.

In [ ]:
# pista: credito['incumplimiento'].value_counts(normalize=True)

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio C2 · Preparar y comparar clasificadores

Creen un `ClassificationExperiment` con target `incumplimiento`, `session_id=42`, 70% de
entrenamiento, 5 pliegues **estratificados** y variables **normalizadas**. Comparen `dummy`, `lr`, `dt`,
`rf` y `gbc` ordenando por **AUC**, y guarden el mejor en `mejor_clf`. ¿Qué modelo gana? ¿Cuánto mejora
al `dummy` en AUC y en recall?

**Criterio de éxito:** una tabla ordenada por AUC donde se vea la línea base ingenua y una comparación explícita contra ella.

In [ ]:
# pista:
# from pycaret.tasks import ClassificationExperiment
# exp_clf = ClassificationExperiment(target='incumplimiento', session_id=42, train_size=0.7, fold=5,
#                                    normalize=True, fold_strategy='stratifiedkfold').fit(credito)
# res_clf = exp_clf.compare_models(include=['dummy', 'lr', 'dt', 'rf', 'gbc'], sort='AUC')
# res_clf.leaderboard     y     mejor_clf = res_clf.best

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio C3 · La matriz de confusión en lenguaje de negocio

Grafiquen la **matriz de confusión** de `mejor_clf`. Identifiquen los **falsos negativos** y los
**falsos positivos** y expliquen qué significa cada uno para el banco. ¿Cuál de los dos errores es más
caro?

**Criterio de éxito:** nombrar cada celda de la matriz como una situación del banco y justificar cuál error cuesta más.

In [ ]:
# pista: exp_clf.plot_model(mejor_clf, plot='confusion_matrix')

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio C4 · Mover el umbral según el costo de los errores

Primero miren el gráfico de **umbral** de `mejor_clf`. Luego, con las predicciones de prueba, armen la
tabla de aciertos y errores para el umbral **0,5** y para el umbral **0,3**. Suponiendo que un falso
negativo cuesta **5** y un falso positivo cuesta **1**, calculen el **costo total** de cada umbral (pueden
hacer la cuenta a mano). ¿Cuál recomiendan?

**Criterio de éxito:** dos tablas (una por umbral), el costo de cada una y una recomendación justificada con el costo.

In [ ]:
# pista:
# exp_clf.plot_model(mejor_clf, plot='threshold').show()
# p = exp_clf.predict_model(mejor_clf).predictions   # 'prediction_score' = probabilidad de incumplir
# p['rechazar'] = (p['prediction_score'] >= 0.5).astype(int)
# pd.crosstab(p['incumplimiento'], p['rechazar'])     # repitan cambiando 0.5 por 0.3

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque D · Clase 5: no supervisado y segmentación

**Caso:** los clientes de telecom de la Clase 5 (`clientes.csv`), **sin la columna `fuga`**: no hay
respuesta correcta que predecir. Esta vez segmentamos con **cuatro** variables numéricas, incluida la
**edad**, que en clase no se usó. Dejamos `plan` fuera del agrupamiento para usarla después, al perfilar.

In [ ]:
clientes = pd.read_csv(BASE + 'clientes.csv').drop(columns='fuga')
variables = ['edad', 'antiguedad_meses', 'cargo_mensual', 'llamadas_soporte']
X = clientes[variables]
print('Clientes x variables:', X.shape)
X.head()

### Ejercicio D1 · Por qué estandarizar

Miren el resumen estadístico de `X` (mínimo, máximo y desviación). ¿Qué variable dominaría el
agrupamiento si **no** estandarizan? Luego creen un `ClusteringExperiment` con `session_id=42` y
`normalize=True`.

**Criterio de éxito:** señalar qué variable dominaría y por qué, y dejar creado el experimento.

In [ ]:
# pista:
# X.describe()
# from pycaret.tasks import ClusteringExperiment
# exp_clu = ClusteringExperiment(session_id=42, normalize=True).fit(X)

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D2 · Elegir el número de grupos (k)

Esta celda **ya viene escrita**: ejecútenla. Calcula, para k = 2 a 8, la **inercia** (método del codo)
y el **silhouette** sobre los datos estandarizados. Lean los dos gráficos y la tabla: ¿qué k proponen?
Recuerden que el criterio final es que los grupos sean **accionables**.

**Criterio de éxito:** un k propuesto con una razón que combine el codo, el silhouette y el negocio.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X)        # estandarizamos igual que en normalize=True
ks = list(range(2, 9))
inercia, silueta = [], []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)   # probamos 10 inicios y nos quedamos con el mejor
    inercia.append(km.inertia_)                       # dispersión interna: siempre baja al agregar grupos
    silueta.append(silhouette_score(Xs, km.labels_))  # qué tan separado queda cada cliente de los otros grupos

px.line(x=ks, y=inercia, markers=True, title='Método del codo (Inercia según k)',
        labels={'x': 'Número de grupos (k)', 'y': 'Inercia'}).show()
px.line(x=ks, y=silueta, markers=True, title='Coeficiente de Silhouette según k',
        labels={'x': 'Número de grupos (k)', 'y': 'Silhouette'}).show()
pd.DataFrame({'k': ks, 'inercia': np.round(inercia), 'silhouette': np.round(silueta, 3)})

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D3 · Segmentar, perfilar y nombrar

Con el k que eligieron, creen el modelo `kmeans`, asignen el segmento a cada cliente y armen el
**perfil** de cada grupo (promedios). Luego cuenten cuántos clientes de cada **plan** hay en cada
segmento. Pongan un **nombre** de negocio a cada grupo y propongan **una acción** por segmento.

**Criterio de éxito:** una tabla de perfiles, un nombre y una acción por segmento, apoyados en los promedios.

In [ ]:
# pista (cambien 4 por el k que eligieron):
# res_clu = exp_clu.create_model('kmeans', n_clusters=4)    # en PyCaret 4 se usa n_clusters
# segmentos = exp_clu.assign_model(res_clu.pipeline)       # agrega la columna 'Cluster'
# segmentos.groupby('Cluster')[variables].mean()
# pd.crosstab(segmentos['Cluster'], clientes['plan'])

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D4 · Ver los segmentos en 2D

Resuman las cuatro variables estandarizadas (`Xs`, de D2) en **dos componentes principales** con PCA
y dibujen un gráfico de dispersión coloreado por segmento. ¿Se ven separados los grupos? ¿Qué cuidado hay
que tener al leer esos ejes?

**Criterio de éxito:** un gráfico 2D coloreado por segmento, leído con sus límites.

In [ ]:
# pista (como en la Clase 5):
# from sklearn.decomposition import PCA
# Z = PCA(n_components=2).fit_transform(Xs)
# px.scatter(x=Z[:, 0], y=Z[:, 1], color=segmentos['Cluster'])

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D5 · Clientes fuera del patrón

Creen un `AnomalyExperiment` con `session_id=42` sobre `X` y un modelo `iforest` que marque al **3%**
de los clientes como anómalos. Muestren los **5 más raros** y el **% de anómalos por plan**. ¿Qué harían
con esa lista? ¿Significa que son fraudes?

**Criterio de éxito:** la lista de los más raros, el % por plan y una acción prudente que incluya revisión humana.

In [ ]:
# pista:
# from pycaret.tasks import AnomalyExperiment
# ax = AnomalyExperiment(session_id=42).fit(X)
# iso = ax.create_model('iforest', fraction=0.03)
# raros = ax.assign_model(iso.pipeline)          # agrega 'Anomaly' (1 = raro) y 'Anomaly_Score'
# raros['plan'] = clientes['plan']
# raros.sort_values('Anomaly_Score', ascending=False).head(5)
# raros.groupby('plan')['Anomaly'].mean() * 100

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

---

**Síntesis.** En el bloque A conocieron y prepararon una tabla (tipos, faltantes, outliers y fuga de
datos). En B compararon modelos de regresión y leyeron el error en UF, también por segmento. En C vieron
que con desbalance el *accuracy* engaña y que el umbral es una decisión de costos. En D descubrieron
segmentos sin etiqueta, los visualizaron y detectaron casos raros para revisión humana. En todos los
bloques, el código es el medio: lo central es **interpretar y decidir**.

Repasen con las **guías de estudio** y las **tarjetas** de las clases 2 a 5.

## Atribución de datos

**Casas de la Región Metropolitana** (`precios_casas_rm.csv`)

**Creador:** Sebastián Contreras, extracción y preparación para uso docente.
**Fuente:** avisos inmobiliarios de [Enlace Inmobiliario](https://www.enlaceinmobiliario.cl/) y [GoPlaceIt](https://www.goplaceit.com/), obtenidos mediante *web scraping*.
**Licencia:** No declarada para esta compilación; fuentes originales: [Enlace Inmobiliario](https://www.enlaceinmobiliario.cl/) y [GoPlaceIt](https://www.goplaceit.com/).
**Modificación:** columnas estandarizadas, codificación UTF-8 y faltantes didácticos; no se incluyen identificadores personales.

**Solicitudes de crédito** (`credito_aleman.csv`)

**Creador:** Hans Hofmann (Statlog German Credit Data).
**Fuente:** [UCI Statlog German Credit Data](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data).
**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
**Modificación:** selección de 8 variables, columnas traducidas al español y variables ordinales codificadas como números.

**Clientes de telecom** (`clientes.csv`)

**Creador:** Equipo docente del módulo.
**Procedencia:** Datos generados con [`generar_clientes.py`](../../../scripts/generar_clientes.py), con semilla fija.
**Nota:** Datos sintéticos; no representan a una organización real ni a personas reales.
**Modificación:** en estos ejercicios se excluye la columna `fuga` para el bloque de segmentación.